# DPO Pair Construction

This notebook converts model generations into structured preference pairs suitable for Direct Preference Optimization (DPO).

Given multiple candidate outputs per prompt, we:

- Evaluate each output using a judge model
- Categorize outputs as positive, negative, or unknown
- Construct (chosen, rejected) pairs
- Prepare the dataset in DPO-compatible format

This stage transforms raw model generations from repeated and hint sampling into supervised preference signals.


In [ ]:
import os
import json
from datasets import Dataset, DatasetDict

from utils.dpo_helpers import (
    load_jsonl,
    split_positive_negative,
    construct_dpo_pairs,
    domain_split,
)


In [ ]:
import os
from pathlib import Path


# Resolve Repository Root
try:
    REPO_ROOT = Path(__file__).resolve().parents[1]
except NameError:
    REPO_ROOT = Path.cwd()

# Dataset Folder (must match previous notebooks)
DATA_FOLDER_NAME = "data_sky"   # <-- change if needed

# Outputs from inference step
OUT_DIR = REPO_ROOT / f"outputs_{DATA_FOLDER_NAME}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# JSONL guide files
GUIDE_PATH = OUT_DIR / "guide.jsonl"
GUIDE_REV_PATH = OUT_DIR / "guide_reverse.jsonl"
BEST_OF_N_PATH = OUT_DIR / "best_of_n.jsonl"

# DPO dataset save directory
SAVE_DIR = REPO_ROOT / f"dpo_dataset_{DATA_FOLDER_NAME}"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using outputs directory: {OUT_DIR}")
print(f"DPO dataset will be saved to: {SAVE_DIR}")


In [ ]:
guide = load_jsonl(GUIDE_PATH)
guide_rev = load_jsonl(GUIDE_REV_PATH)
outputs = load_jsonl(BEST_OF_N_PATH)

print(len(guide), len(guide_rev), len(outputs))

### Categorizing Generated Outputs

For each prompt, generated responses are classified into:

1. Positive — judge agrees with ground truth
2. Negative — judge disagrees
3. Unknown — parsing failed or ambiguous

This categorization enables structured construction of preference pairs while quantifying uncertainty (unknown rate).

Robust filtering is essential to avoid injecting noisy supervision into DPO training.

In [ ]:
guide_split = split_positive_negative(guide)
guide_rev_split = split_positive_negative(guide_rev)
output_split = split_positive_negative(outputs)

### DPO Pairs Construction
We construct DPO training pairs following the formulation introduced by Rafailov et al. (2023).

Pairs are created using two strategies:

1. Repeated Sampling

- Positive generations are paired against negative generations from the same prompt.
- This leverages stochastic diversity to construct meaningful preference comparisons.

2. Hint Sampling (Guide vs Guide-Reverse)

- We compare outputs generated under original and reversed hint conditioning.
-  This helps control for positional bias and strengthens supervision robustness.

Importantly, each judge output contains structured supervision of the form:

```{"reason": "...", "better_answer": 1}```


This provides not only a binary preference signal (better_answer), but also an explicit reasoning trace (reason) explaining the judgment.

By preserving both preference labels and reasoning, we obtain richer supervision signals that improve interpretability and reduce ambiguity in downstream DPO training.

In [ ]:
dpo_pairs = construct_dpo_pairs(
    guide_split=guide_split,
    guide_rev_split=guide_rev_split,
    output_split=output_split,
)

dataset = Dataset.from_dict(dpo_pairs)
print("Total DPO pairs:", len(dataset))

In [ ]:
split = dataset.train_test_split(test_size=0.1, seed=42)

dataset_dict = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
})

In [ ]:
domain_splits = domain_split(dataset)

In [ ]:
dataset_dict.save_to_disk(SAVE_DIR)

meta_info = {
    "total_pairs": len(dataset),
    "domains": list(set(dataset["tag"])),
    "pair_types": list(set(dataset["pair_type"])),
}

with open(os.path.join(SAVE_DIR, "metadata.json"), "w") as f:
    json.dump(meta_info, f, indent=2)

for domain, split in domain_splits.items():
    path = os.path.join(SAVE_DIR, f"domain_{domain}")
    os.makedirs(path, exist_ok=True)
    split["train"].save_to_disk(os.path.join(path, "train"))
    split["test"].save_to_disk(os.path.join(path, "validation"))

In [ ]:
print("DPO dataset saved successfully")
print("Train samples:", len(dataset_dict["train"]))
print("Validation samples:", len(dataset_dict["validation"]))
print("Metadata saved at:", f"{SAVE_DIR}/metadata.json")

### References

- Rafailov et al., 2023. Direct Preference Optimization: Your Language Model is Secretly a Reward Model.
https://arxiv.org/abs/2305.18290
- Ye, Z., et al., (2025).
Learning LLM-as-a-Judge for Preference Alignment.
https://openreview.net/forum?id=HZVIQE1MsJ